In [37]:
# import packages 
import pandas as pd 

In [38]:
# Import all geneids for DMRs
genes_ar = pd.read_csv('/home/maggy/PhD/WholeGenomeBisulphiteSequencing/DNAseq_pipeline/7_match_dmr_to_gene/dmr_within_geneborders_genelist/dmr_withinGeneBorder_AvsR_geneids.csv')
genes_na = pd.read_csv('/home/maggy/PhD/WholeGenomeBisulphiteSequencing/DNAseq_pipeline/7_match_dmr_to_gene/dmr_within_geneborders_genelist/dmr_withinGeneBorder_NvsA_geneids.csv')
genes_nr = pd.read_csv('/home/maggy/PhD/WholeGenomeBisulphiteSequencing/DNAseq_pipeline/7_match_dmr_to_gene/dmr_within_geneborders_genelist/dmr_withinGeneBorder_NvsR_geneids.csv')
# merge to one df and drop duplicates 
genes = pd.concat([genes_na, genes_nr, genes_ar])
genes = genes.drop_duplicates()
genes.to_csv('2026_03_04_dmr_geneids.csv', index=False)

In [39]:
genes

,geneid
0,ccar_ua02-g2350
1,ccar_ua05-g5816
2,ccar_ua10-g10025
3,ccar_ua15-g14205
4,ccar_ua15-g14343
5,ccar_ua16-g15662
6,ccar_ua18-g16750
7,ccar_ub04-g26931
8,ccar_ub09-g32086
9,ccar_ub11-g34027


In [40]:
# import ccar_annotation_v5.interproscan
go = pd.read_csv('ccar_annotation_v5.interproscan.GO.tsv', sep='\t', header=None)
go.rename(columns={0:'geneid', 1:'go_terms'}, inplace=True)
go['geneid'] = go['geneid'].str.split('.').str[0]
go_dmr_filtered = go.loc[go['geneid'].isin(set(genes['geneid']))]
len(set(go_dmr_filtered['go_terms']))
print(set(go_dmr_filtered['go_terms']))
go_dmr_filtered.to_csv('2026_03_04_interproscan_go.csv')

{'GO:0016491', 'GO:0005216|GO:0006811|GO:0016020|GO:0055085', 'GO:0005737|GO:0016627', 'GO:0004252|GO:0006508', 'GO:0016790', 'GO:0008270', 'GO:0003676|GO:0006355', 'GO:0004930|GO:0007186', 'GO:0016020', 'GO:0008094', 'GO:0003824', 'GO:0051536', 'GO:0004252', 'GO:0003676', 'GO:0008237', 'GO:0005245|GO:0005891|GO:0070588', 'GO:0000981|GO:0006355', 'GO:0005525', 'GO:0003723', 'GO:0051260', 'GO:0016538|GO:0044772', 'GO:0005509', 'GO:0061630|GO:0071596', 'GO:0005515', 'GO:0003700|GO:0006357', 'GO:0005102|GO:0030155|GO:0030334|GO:0045995', 'GO:0003677', 'GO:0006486|GO:0008373', 'GO:0006508|GO:0008236', 'GO:0003924|GO:0005525', 'GO:0005509|GO:0007155', 'GO:0007186|GO:0016020', 'GO:0003700|GO:0006355|GO:0043565', 'GO:0004930|GO:0007186|GO:0016020', 'GO:0005524', 'GO:0004222|GO:0006508', 'GO:0007155', 'GO:0005869|GO:0007017', 'GO:0005634', 'GO:0005509|GO:0005576|GO:0007155', 'GO:0016887', 'GO:0030163'}


In [41]:
go_dmr_filtered

,geneid,go_terms
3713,ccar_ua02-g1919,GO:0005515
12081,ccar_ua05-g5120,GO:0005869|GO:0007017
12082,ccar_ua05-g5120,GO:0005869|GO:0007017
12771,ccar_ua05-g5369,GO:0005524
12772,ccar_ua05-g5369,GO:0008094
...,...,...
107204,ccar_ub23-g43991,GO:0003723
107205,ccar_ub23-g43991,GO:0003676
107206,ccar_ub23-g43991,GO:0003723
107207,ccar_ub23-g43991,GO:0003676


In [42]:
# import ccar_annotation_v5.interproscan.tsv
scan = pd.read_csv('ccar_annotation_v5.interproscan.tsv', sep='\t',header=None)
scan.rename(columns={0:'geneid'}, inplace=True)
scan['geneid'] = scan['geneid'].str.split('.').str[0]
scan_dmr_filtered = scan.loc[scan['geneid'].isin(set(genes['geneid']))]
scan_dmr_filtered.to_csv('2026_03_04_interproscan.csv')

In [43]:
# import user_ko_definition.txt
kegg = pd.read_csv('user_ko_definition.txt', sep='\t', header=None)
kegg.rename(columns={0:'geneid'}, inplace=True)
kegg['geneid'] = kegg['geneid'].str.split('.').str[0]
kegg_dmr_filtered = kegg.loc[kegg['geneid'].isin(set(genes['geneid']))]
kegg_dmr_filtered.to_csv('2026_03_04_ko_definition.csv')

# Create Supplementary Table 

In [ ]:
# set gene id as index for easier merging 
go_dmr_filtered.set_index('geneid', inplace=True)

# import protein names and protein function description 
names = pd.read_csv('/home/maggy/PhD/WholeGenomeBisulphiteSequencing/DNAseq_pipeline/13_protein_sequence/blastp_hits_swissprot_genes.txt', sep='\t', index_col=0)

# add protein name information 
summary_table = pd.merge(names, go_dmr_filtered, left_index=True, right_index=True, how='inner')

# import go names: GO term, description, and biological function 
go_names = pd.read_csv('/home/maggy/PhD/WholeGenomeBisulphiteSequencing/RNAseq_pipeline/14_functional_annotation/go_data.txt', sep='\t')


In [ ]:
# Build lookup dicts from go_names
id_to_name = dict(zip(go_names['id'], go_names['name']))
id_to_ns   = dict(zip(go_names['id'], go_names['namespace']))

def map_go_terms_to_desc_and_ns(go_str):
    if pd.isna(go_str):
        return pd.Series({'go_description': pd.NA, 'go_namespace': pd.NA})
    ids = go_str.split('|')
    descs = [id_to_name.get(go_id, '') for go_id in ids]
    nss   = [id_to_ns.get(go_id, '')   for go_id in ids]
    # join back with '|' like the original column
    return pd.Series({
        'go_description': '|'.join(descs),
        'go_namespace':   '|'.join(nss),
    })

# Apply to your summary_table
summary_table[['go_description', 'go_namespace']] = summary_table['go_terms'].apply(map_go_terms_to_desc_and_ns)


In [ ]:
# export 
summary_table.to_csv('dmr_annotation_overview.csv')